In [1]:
# 动作类型列表
ACTOR_TYPES_str = [
    "AircraftTakeOffActor",
    "ReturnToBaseActor",
    "WayPointMoveActor",
    "MobilityActor",
    "AttackTargetActor",
    "SensorControlActor",
    "DeploySonobuoyActor",
    "CancelAttackActor"
]

def update_entity_action_frame_link(entity_action_frame_link, execute_result, step):
    for entity_id, action_is_performs in execute_result.items():
        # 找出执行的动作 ID
        performed_action_ids = [i for i, is_performed in enumerate(action_is_performs) if is_performed]

        # 如果实体 ID 不在结果字典中，初始化一个空列表
        if entity_id not in entity_action_frame_link:
            entity_action_frame_link[entity_id] = []

        # 添加当前步骤的执行动作 ID 列表
        entity_action_frame_link[entity_id].append({step: performed_action_ids})

    return entity_action_frame_link

# 初始化实体 ID: 动作 - 帧数链路
entity_action_frame_link = {}

# 模拟逐帧添加 execute_result
execute_results = [
    {
        1: [True, False, True, False, False, True, False, False],
        2: [False, True, False, False, True, False, False, False]
    },
    {
        1: [False, False, False, True, False, False, True, False],
        2: [True, False, False, False, False, False, False, True]
    }
]

for step, execute_result in enumerate(execute_results):
    entity_action_frame_link = update_entity_action_frame_link(entity_action_frame_link, execute_result, step)

print(entity_action_frame_link)

{1: [{0: [0, 2, 5]}, {1: [3, 6]}], 2: [{0: [1, 4]}, {1: [0, 7]}]}


In [21]:


import numpy as np

# 潜艇  船  武器  飞机
base_find_event_reward = np.array([
    [1.2, 1.1, 1.0, 0.8],  # AircraftTakeOffActor 飞机起飞
    [0, 0.1, 0.1, 0.1],  # ReturnToBaseActor
    [2.0, 1.5, 1.0, 0.8],  # WayPointMoveActor 航线机动
    [3.0, 2.0, 1.0, 1.0],  # MobilityActor
    [0.0, 0.0, 0.0, 0.0],  # AttackTargetActor
    [2.5, 2.0, 1.5, 1.0],  # SensorControlActor
    [3.0, 2.0, 1.5, 1.5],  # DeploySonobuoyActor
    [0.0, 0.0, 0.0, 0.0]  # CancelAttackActor
]) * 0.8


class SomeClass:
    def update_detection_report_reward(self, maddpg, entity_action_frame_link, target_types, detector_ids, detector_types, detect_step,
                                       is_sonobuoy):
        # 动作类型索引
        ACTION_TYPES = {
            "AircraftTakeOffActor": 0,
            "ReturnToBaseActor": 1,
            "WayPointMoveActor": 2,
            "MobilityActor": 3,
            "AttackTargetActor": 4,
            "SensorControlActor": 5,
            "DeploySonobuoyActor": 6,
            "CancelAttackActor": 7
        }

        for i in range(len(is_sonobuoy)):
            if is_sonobuoy[i]:
                # 处理声呐浮标发现的情况（原有逻辑）
                detector_id = detector_ids[i]
                detect_frame = detect_step[i]
                target_type_index = target_types[i]

                # 对该帧的声呐浮标动作进行奖励
                maddpg.buffer[detect_frame][ACTION_TYPES["DeploySonobuoyActor"]] += base_find_event_reward[
                    ACTION_TYPES["DeploySonobuoyActor"]][target_type_index]

                # 通过 entity_action_frame_link 找到该实体对应的部署声呐浮标帧数往前推
                action_list = entity_action_frame_link.get(detector_id, [])
                deploy_index = None
                for idx, frame_info in enumerate(action_list):
                    frame = list(frame_info.keys())[0]
                    if frame == detect_frame and ACTION_TYPES["DeploySonobuoyActor"] in frame_info[frame]:
                        deploy_index = idx
                        break

                if deploy_index is not None:
                    last_action_frame = {}
                    to_break = False
                    for j in range(deploy_index - 1, -1, -1):
                        frame_info = action_list[j]
                        frame = list(frame_info.keys())[0]
                        actions = frame_info[frame]

                        for action in actions:
                            if action not in last_action_frame:
                                last_action_frame[action] = frame
                                if action == ACTION_TYPES["AircraftTakeOffActor"]:
                                    # 对起飞进行奖励
                                    maddpg.buffer[frame][action] += base_find_event_reward[action][target_type_index]
                                    # 对起飞帧内的其他动作进行奖励
                                    for other_action in actions:
                                        if other_action != action and other_action not in last_action_frame:
                                            last_action_frame[other_action] = frame
                                            maddpg.buffer[frame][other_action] += base_find_event_reward[other_action][
                                                target_type_index]
                                    to_break = True
                                    break
                                elif action != ACTION_TYPES["DeploySonobuoyActor"]:
                                    # 对其他动作类型进行奖励
                                    maddpg.buffer[frame][action] += base_find_event_reward[action][target_type_index]
                        if to_break:
                            break
            else:
                # 处理非声呐浮标发现的情况
                detector_id = detector_ids[i]
                detect_frame = detect_step[i]
                target_type_index = target_types[i]

                # 找到该实体在 detect_frame 之前最近的有效帧（允许包含声呐浮标动作，但需有其他动作）
                action_list = entity_action_frame_link.get(detector_id, [])
                last_valid_frame = None
                last_valid_actions = []

                # 从后往前遍历，找到第一个在 detect_frame 之前且至少有一个非声呐浮标动作的帧
                for frame_info in reversed(action_list):
                    frame = list(frame_info.keys())[0]
                    if frame >= detect_frame:
                        continue  # 跳过当前帧及之后的帧
                    actions = frame_info[frame]
                    # 检查是否有非声呐浮标动作
                    if any(action != ACTION_TYPES["DeploySonobuoyActor"] for action in actions):
                        last_valid_frame = frame
                        last_valid_actions = actions
                        break

                if last_valid_frame is not None:
                    # 对有效帧中的非声呐浮标动作进行奖励
                    for action in last_valid_actions:
                        if action != ACTION_TYPES["DeploySonobuoyActor"]:
                            maddpg.buffer[last_valid_frame][action] += base_find_event_reward[action][target_type_index]


# 以下是简单的测试代码
class MADDPG:
    def __init__(self, num_frames):
        self.buffer = np.zeros((num_frames, 8))


if __name__ == "__main__":
    maddpg = MADDPG(num_frames=10)
    entity_action_frame_link = {
        1: [
            {0: [0, 5]},   # 起飞 + 传感器控制
            {1: [2, 5]},   # 航线机动 + 传感器控制
            {2: [3, 5]},   # 移动 + 传感器控制
            {4: [6, 5]},   # 部署声呐浮标 + 传感器控制
            {6: [6, 2, 3]},# 部署声呐浮标 + 航线机动 + 移动
        ]
    }
    target_types = [0, 1]
    detector_ids = [1, 1]
    detector_types = ["sonobuoy", "radar"]
    detect_step = [4, 6]
    is_sonobuoy = [True, False]  # 第二例为非声呐浮标发现

    obj = SomeClass()
    obj.update_detection_report_reward(maddpg, entity_action_frame_link, target_types, detector_ids, detector_types, detect_step,
                                       is_sonobuoy)
    print(maddpg.buffer)



[[0.96 0.   0.   0.   0.   0.   0.   0.  ]
 [0.   0.   1.6  0.   0.   0.   0.   0.  ]
 [0.   0.   0.   2.4  0.   2.   0.   0.  ]
 [0.   0.   0.   0.   0.   0.   0.   0.  ]
 [0.   0.   0.   0.   0.   1.6  2.4  0.  ]
 [0.   0.   0.   0.   0.   0.   0.   0.  ]
 [0.   0.   0.   0.   0.   0.   0.   0.  ]
 [0.   0.   0.   0.   0.   0.   0.   0.  ]
 [0.   0.   0.   0.   0.   0.   0.   0.  ]
 [0.   0.   0.   0.   0.   0.   0.   0.  ]]


In [24]:
import numpy as np

from typing import List

# 模拟输入数据
encoded_state = np.zeros((1, 5, 29))  # batch_size=1, 5个实体, 29维特征
state_mask = np.array([[1, 1, 1, 1, 0]])  # 最后一个实体是padding

# 设置实体类型（第13维）
encoded_state[0, 0, 13] = 0  # 实体0: Aircraft
encoded_state[0, 1, 13] = 1  # 实体1: Ship 
encoded_state[0, 2, 13] = 2  # 实体2: Submarine
encoded_state[0, 3, 13] = 3  # 实体3: Facility (将被过滤)

def modify_action_masks(encoded_state, state_mask):
    # 模拟的get_env_entity_ids函数
    def get_env_entity_ids(state, mask, target_side, allowed_types):
        entity_indices = []
        for i in range(state.shape[1]):
            if mask[0, i] == 1 and state[0, i, 13] in allowed_types:
                entity_indices.append(i)
        return entity_indices
    
    # 获取有效实体索引 (类型0/1/2)
    state_np = encoded_state if isinstance(encoded_state, np.ndarray) else encoded_state.cpu().numpy()
    mask_np = state_mask if isinstance(state_mask, np.ndarray) else state_mask.cpu().numpy()
    active_entity_indices = get_env_entity_ids(state_np, mask_np, 
                                             target_side=1, 
                                             allowed_types=[0, 1, 2])
    
    # 初始化动作掩码 (8个动作类型 x 最大实体数)
    ACTOR_TYPES = list(range(8))  # 简化表示
    MAX_ACTION_ENTITIES = 50
    action_masks = np.zeros((len(ACTOR_TYPES), MAX_ACTION_ENTITIES), dtype=np.float32)
    
    # 遍历每个有效实体设置权限
    for pos_idx, entity_idx in enumerate(active_entity_indices):
        entity_type = state_np[0, entity_idx, 13]
        
        if entity_type == 0:  # Aircraft
            allowed_actions = [0,1,2,3,4,5,6,7]  # 全部允许
        elif entity_type == 1:  # Ship
            allowed_actions = [2,3,4,5,7]  # 不能起飞/返航/部署声呐
        elif entity_type == 2:  # Submarine
            allowed_actions = [2,3,4,5,7]  # 同Ship
        
        # 设置允许的动作位
        for action_idx in allowed_actions:
            action_masks[action_idx, pos_idx] = 1
            
    return action_masks[:, :len(active_entity_indices)]  # 只返回有效部分

# 执行示例
action_masks = modify_action_masks(encoded_state, state_mask)
print("动作掩码矩阵 (动作类型 x 有效实体):")
print(action_masks)

动作掩码矩阵 (动作类型 x 有效实体):
[[1. 0. 0.]
 [1. 0. 0.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 0. 0.]
 [1. 1. 1.]]


In [ ]:
import numpy as np
buffer = [[{"rewards": np.zeros(3)}]]
latest = buffer[-1]
frame = latest[0]

# 修改某个 reward 项
frame["rewards"][1] += 5

print(buffer[0][0]["rewards"])  # 输出：[0. 5. 0.] ✅ 改了

In [5]:
import math
def _bearing(lon1, lat1, lon2, lat2):
    """返回两个经纬度之间的正北方向方位角（弧度）"""
    # 转为弧度
    lon1, lat1, lon2, lat2 = map(math.radians, [lon1, lat1, lon2, lat2])
    
    dlon = lon2 - lon1
    x = math.sin(dlon) * math.cos(lat2)
    y = math.cos(lat1)*math.sin(lat2) - math.sin(lat1)*math.cos(lat2)*math.cos(dlon)
    angle = math.atan2(x, y)
    return angle % (2 * math.pi)  # 保证结果在 [0, 2π)
_bearing(0,0,50,50)

0.5712882779112652

In [29]:
import numpy as np
from gym import spaces
import torch
import torch.nn as nn
import torch.nn.functional as F

from entity import  AIR_STATUS_MAP, MAX_ACTION_ENTITIES, MAX_TARGETS
# 所有actor指令输出横向拼接 最大维度为5


# ACTOR_TYPES_str = [
#     AircraftTakeOffActor,0
#     ReturnToBaseActor, 1
#     WayPointMoveActor,2
#     MobilityActor, 3 
#     AttackTargetActor,4 
#     SensorControlActor, 5
#     DeploySonobuoyActor, 6 
#     CancelAttackActor 7
# ]

def haversine(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(torch.deg2rad, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = torch.sin(dlat/2)**2 + torch.cos(lat1) * torch.cos(lat2) * torch.sin(dlon/2)**2
    c = 2 * torch.arcsin(torch.sqrt(a))
    return c * 6371  # 地球半径6371km
class AttackTargetActor(nn.Module):
    
    def __init__(self, 
                 self_obs_dim=512,    # 攻击方自身状态维度
                 target_obs_dim=256): # 目标状态特征维度
        super().__init__()
        
        # 攻击方自身状态编码器
        self.self_encoder = nn.Sequential(
            nn.Linear(self_obs_dim, 256),
            nn.LayerNorm(256),
            nn.ReLU()
        )
        
        # 目标特征适配器（当目标特征维度需要调整时）
        self.target_adapter = nn.Sequential(
            nn.Linear(target_obs_dim, 256),
            nn.LayerNorm(256),
            nn.ReLU()
        )
        
        # 攻击决策网络
        self.attack_net = nn.Sequential(
            nn.Linear(256 + 256 + 1, 128),  # 增加距离特征
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

        self.perform_net = nn.Sequential(
            nn.Linear(256, 128), 
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )
    
    def forward(self, state, target_features, target_mask=None, self_mask=None, self_coords=None, target_coords=None):
        # 计算球面距离
        # self_coords [batch, max_entity_len, 2]
        # target_coords [batch,target_max_entity_len, 2]
        if self_coords is not None and target_coords is not None:
            # 扩展广播形状 [B, E, T, 2]

            self_coords_exp = self_coords.unsqueeze(2)  # [B,E,1,2]
            target_coords_exp = target_coords.unsqueeze(1)  # [B,1,T,2]
            
            # 计算球面距离
            distance = haversine(
                self_coords_exp[...,0], 
                self_coords_exp[...,1],
                target_coords_exp[...,0],
                target_coords_exp[...,1]
            )  # [B,E,T]
            
            # 距离惩罚系数，可调节强度（越大越偏向近处）
            distance_penalty = -0.01 * distance  # [B, E, T]
        else:
            distance_penalty = 0

        # 编码自身状态（每个实体）
        self_feat = self.self_encoder(state)  # [batch, max_entity_len, 256]
        
        # 处理目标特征
        target_feat = self.target_adapter(target_features)  # [batch, max_targets, 256]
        
        # 特征融合（实体与目标交叉组合）
        self_exp = self_feat.unsqueeze(2).expand(-1,-1,target_feat.size(1),-1)  # [B,E,T,256]
        target_exp = target_feat.unsqueeze(1).expand(-1,self_feat.size(1),-1,-1)  # [B,E,T,256]
        
        # 加入距离特征
        fused = torch.cat([
            self_exp, 
            target_exp,
            distance.unsqueeze(-1)  # [B,E,T,1]
        ], dim=-1)  # [B,E,T,513]
        
        # 计算攻击概率
        attack_logits = self.attack_net(fused).squeeze(-1)  # [B,E,T]
        attack_probs = torch.softmax(attack_logits + distance_penalty, dim=-1)
        
        # 应用双重掩码
        if target_mask is not None:
            target_mask = target_mask.unsqueeze(1).expand_as(attack_probs)
            attack_probs = attack_probs.masked_fill(~target_mask, 0)
        if self_mask is not None:
            self_mask = self_mask.unsqueeze(-1).expand_as(attack_probs)
            attack_probs = attack_probs.masked_fill(~self_mask, 0)
        
        # 执行概率
        perform_probs = self.perform_net(self_feat).squeeze(-1)  # [B,E]
        
        return {
            "selected_index": torch.argmax(attack_probs, dim=-1),
            "attack_prob": perform_probs
        }

import torch
import numpy as np

# 生成10个测试样本
test_cases = [
    # 格式：(自方坐标, 目标坐标列表, 案例描述)
    ([[0.0, 0.0]], [[0.1,0.1], [1.0,1.0], [2.0,2.0]], "1近2远"),
    ([[22.5, 114.0]], [[22.4,114.0], [23.0,114.5]], "香港附近海域"),  # 香港周边
    ([[-33.9, 151.2]], [[-37.8,151.1], [-34.0,151.3]], "悉尼港"),  # 悉尼
    ([[35.7, 139.7]], [[35.6,139.6], [35.8,139.8]], "东京湾"),  # 东京
    ([[37.8, -122.4]], [[37.7,-122.3], [38.0,-122.5]], "旧金山湾"),  # 旧金山
    ([[41.9, 12.5]], [[41.8,12.4], [42.0,12.6]], "罗马附近"),  # 罗马
    ([[48.9, 2.3]], [[48.8,2.2], [49.0,2.4]], "巴黎周边"),  # 巴黎
    ([[55.8, 37.6]], [[55.7,37.5], [56.0,37.7]], "莫斯科区域"),  # 莫斯科
    ([[-23.5, -46.6]], [[-23.4,-46.5], [-23.6,-46.7]], "圣保罗近郊"),  # 圣保罗
    ([[31.2, 121.5]], [[35.1,121.4], [31.3,121.6]], "长江口"),  # 上海
]

for i, (self_pos, target_pos_list, desc) in enumerate(test_cases):
    print(f"\n=== 测试案例 {i+1}: {desc} ===")
    
    # 转换坐标到tensor
    self_coords = torch.tensor([self_pos], dtype=torch.float32)  # [1,1,2]
    target_coords = torch.tensor([target_pos_list], dtype=torch.float32)  # [1,T,2]
    
    # 生成随机但相同的特征
    batch_size = 1
    num_entities = 1
    num_targets = len(target_pos_list)
    
    # 使用相同随机种子保证特征一致

    self_obs = torch.randn(batch_size, num_entities, 512)
    target_features = torch.randn(batch_size, num_targets, 256)
    target_features[:] = target_features[:,0:1,:].clone()  # 使所有目标特征相同
    
    # 运行模型
    model = AttackTargetActor()
    with torch.no_grad():
        output = model(
            state=self_obs,
            target_features=target_features,
            self_coords=self_coords,
            target_coords=target_coords
        )
    
    # 解析输出
    selected = output["selected_index"].squeeze().tolist()  # [B,E]->标量
    
    # 计算实际距离
    distances = []
    for t_pos in target_pos_list:
        dist = haversine(
            torch.tensor(self_pos[0][0]),
            torch.tensor(self_pos[0][1]),
            torch.tensor(t_pos[0]),
            torch.tensor(t_pos[1])
        ).item()
        distances.append(f"{dist:.2f}km")
    
    # 打印结果
    print(f"目标距离: {distances}")
    print(f"选择目标: 索引{selected} (最近的是索引{np.argmin([float(d[:-2]) for d in distances])})")
    print("概率分布:", output["attack_prob"].squeeze().tolist())



=== 测试案例 1: 1近2远 ===
目标距离: ['15.73km', '157.25km', '314.47km']
选择目标: 索引0 (最近的是索引0)
概率分布: 0.49173328280448914

=== 测试案例 2: 香港附近海域 ===
目标距离: ['4.52km', '60.10km']
选择目标: 索引0 (最近的是索引0)
概率分布: 0.5343543887138367

=== 测试案例 3: 悉尼港 ===
目标距离: ['379.98km', '14.79km']
选择目标: 索引1 (最近的是索引1)
概率分布: 0.5086733102798462

=== 测试案例 4: 东京湾 ===
目标距离: ['13.98km', '13.99km']
选择目标: 索引0 (最近的是索引0)
概率分布: 0.566203236579895

=== 测试案例 5: 旧金山湾 ===
目标距离: ['12.61km', '16.31km']
选择目标: 索引0 (最近的是索引0)
概率分布: 0.5101435780525208

=== 测试案例 6: 罗马附近 ===
目标距离: ['15.54km', '15.54km']
选择目标: 索引1 (最近的是索引0)
概率分布: 0.5164645314216614

=== 测试案例 7: 巴黎周边 ===
目标距离: ['15.72km', '15.72km']
选择目标: 索引1 (最近的是索引0)
概率分布: 0.5211169123649597

=== 测试案例 8: 莫斯科区域 ===
目标距离: ['14.19km', '20.83km']
选择目标: 索引0 (最近的是索引0)
概率分布: 0.5219908356666565

=== 测试案例 9: 圣保罗近郊 ===
目标距离: ['13.50km', '13.49km']
选择目标: 索引1 (最近的是索引1)
概率分布: 0.5075326561927795

=== 测试案例 10: 长江口 ===
目标距离: ['226.50km', '12.55km']
选择目标: 索引1 (最近的是索引1)
概率分布: 0.5667418241500854
